# 08. LLM Pre-Training: The Foundation of Large Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/The-Pocket/PocketFlow-Tutorial-Video-Generator/blob/main/docs/llm/08_LLM_Pretraining_From_Scratch.ipynb)

> **Tutorial Overview**: Understand self-supervised next-token prediction, tokenization pipelines, causal masking, cross-entropy loss, perplexity, and the full training loop with learning rate scheduling.
> **Original Source**: `pretrain.md`

---


In [ ]:
# Setup & Imports
!pip install -q torch matplotlib numpy

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# **Title: LLM Pre-Training in 30 Minutes**

### Introduction: The Magic is Just Math

You've mastered how a neural network learns. You've seen the elegant dance of gradient descent and backpropagation, the mathematical engine that allows a machine to get progressively better at a task by minimizing its error. But that was with numbers. Clean, predictable, logical numbers.

Now we tackle language.

When a model like ChatGPT writes code, explains quantum physics, or composes a sonnet, it seems like magic. Here's the secret: it's the *exact same learning process*, applied relentlessly to one deceptively simple task: **predict the next token.**

In this tutorial, we will deconstruct the GPT-2 pre-training pipeline step by step. You will learn the exact algorithms and mathematics that allow a machine to teach itself from the raw, unlabeled text of the internet. We will build the entire conceptual pipeline, showing you exactly how "The cat sat on the" becomes a prediction for "mat," and how doing this billions of times creates the emergent intelligence you see today.

We will cover three key stages, anchored by this roadmap:


```mermaid
graph LR
    subgraph "Part 1 & 2: Data Preparation"
        A[Raw Text] --> B{Training Data Algorithm}
        B --> C[Input Token IDs]
    end
    subgraph "Part 3: The Learning Loop"
        C --> D["Model (Embeddings + Transformer)"]
        D --> E[Output Logits]
        E --> F[Softmax -> Probabilities]
        F --> G["Cross-Entropy Loss (Error)"]
        G -.-> D
    end
```


By the end, you will understand:
1.  **Data Preparation:** How we turn the entire internet into an infinite source of free training examples.
2.  **The Learning Loop:** How the model makes a prediction, measures its own error, and systematically corrects itself.

Let's begin by tackling the single biggest bottleneck in the history of AI: data.

## **Part 1: The Data Revolution - Learning Without Labels**

#### The Problem: The Expensive Reality of Supervised Learning

You've seen neural networks master tasks through supervised learning. The recipe is simple: give the model an input (like an image) and a correct output (the label "cat"), and it learns to map one to the other.

But there's a massive bottleneck: getting that labeled training data is painfully expensive and fundamentally limiting. Consider the real-world costs:

*   **Medical Imaging:** Radiologists, who charge hundreds of dollars per hour, are needed to label tumors in MRI scans.
*   **Legal Documents:** Lawyers, billing even more, are required to classify contracts or find evidence in discovery documents.
*   **Scientific Research:** PhD researchers can spend months or years meticulously annotating datasets for their experiments.

This creates two fundamental problems:

1.  **The Scale Ceiling:** The famous ImageNet dataset, with its 14 million labeled images, took years and millions of dollars to create. Yet, this is a tiny fraction of the billions of unlabeled images on the internet that we can't use.
2.  **The "Garbage In, Garbage Out" Problem:** The quality of the model is capped by the quality of its labels. Getting high-quality annotations requires true experts, making the process even more expensive and less scalable.

Supervised learning, for all its power, hits a wall. How do you get to billions or trillions of training examples if every single one requires an expensive human expert?

#### The Breakthrough: The Self-Supervised Engine

The genius of models like GPT-2 wasn't just a bigger architecture—it was abandoning the need for human labels entirely. Instead of asking a human, "What is the right answer?", it asks the text itself.

The task is deceptively simple: **predict the next word.**

That's it. No human annotation is needed. The text provides both the input (the sequence of words so far) and the "label" (the very next word in the sequence). This is the core of **self-supervised learning**.

We promised you an algorithm, and here is the simple, powerful engine that turns any document into an almost unlimited supply of training data.
